In [1]:
import os
import json
# GPU Mode
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import cv2
import numpy as np
import random
import datetime as dt
import matplotlib.pyplot as plt
import math
import tensorflow.keras as tf_keras

# 모델링 도구
from sklearn.model_selection import train_test_split
from tensorflow.keras import mixed_precision
from tensorflow.keras.utils import Sequence, to_categorical
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard, ModelCheckpoint, ReduceLROnPlateau

policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# 장치 확인 (GPU가 잡혀야 정상)
print(f"🔥 현재 TF 버전: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🔥 Docker GPU 연결 성공: {len(gpus)}개 감지됨")
    print(f"   장치명: {gpus[0]}")
else:
    print("❌ GPU 감지 실패 (docker run 할 때 --gpus all 옵션 넣었는지 확인하세요)")

# 시드 고정
seed_constant = 27
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

# 변수 설정
categories = ['신호위반', '중앙선침범', '진로변경위반']
IMAGE_HEIGHT, IMAGE_WIDTH = 128, 128
SEQUENCE_LENGTH = 50

# signal: 신호등 상태 인식, middleLine: 중앙선 침범, whiteLine: 차선 위반
all_classes = ['Signal', 'middleLine', 'whiteLine']

2026-02-10 13:37:09.906830: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-10 13:37:09.906888: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-10 13:37:09.908765: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 5060 Ti, compute capability 12.0
🔥 현재 TF 버전: 2.15.0
🔥 Docker GPU 연결 성공: 1개 감지됨
   장치명: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [2]:
# path 경로에 폴더 없을 경우 새로 생성
def createDirectory(path):
    if not os.path.exists(path):
        os.mkdir(path)

def frame_extraction(folder_path):
    frame_list = []
    # 폴더 경로 + 파일명 합쳐 완전한 경로로 설정 / folder_path 내의 모든 파일과 폴더의 이름을 리스트 형태로
    file_paths = [os.path.join(folder_path, i) for i in os.listdir(folder_path)]

    # 파일을 0 1 바이너리 형태의 넘파이 배열로 읽어와 메모리 내에서 이미지로 디코딩
    for file in file_paths:
        img_array = np.fromfile(file, np.uint8)
        frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)   # 디코딩하여 한글 경로 문제 해결
#         frame = cv2.imread(file)

        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT , IMAGE_WIDTH)) # 64 x 64 px 로 크기 강제 조정
        normalized_frame = resized_frame/255    # 0 ~ 255 사이 픽셀 0 ~ 1 실수로 변경
        frame_list.append(normalized_frame)     # framelist에 전처리된 이미지 적재

        if len(frame_list) == SEQUENCE_LENGTH:  # 프레임 25장 넘어갈 경우 읽기 중단
            break
    if len(frame_list) ==0:                     # 프레임 리스트 내 이미지가 없을 경우 해당 폴더 경로 출력
        print(folder_path)
    if len(frame_list) < SEQUENCE_LENGTH:       # 프레임 전체 수가 25장 이하일 경우
        max_len = len(frame_list)
        point = 0
        while len(frame_list) < SEQUENCE_LENGTH:
            frame_list.append(frame_list[point])    # 맨 처음부터 다시 복사해서 강제로 집어넣기
            point += 1
            if max_len == point:
                point = 0

    return frame_list

In [3]:
def get_data(paths, labels):
    for folder_path, label_index in zip(paths,labels):  # 폴더 경로 리스트와 정답 라벨 리스트 쌍으로 묶기
        feature = frame_extraction(folder_path) # frame_extraction 호출 후 25장 정규화 프레임 가져오기
        feature = np.array(feature) # 리스트 데이터를 모아 넘파이 배열로 수정

        label = np.array([label_index]) # (피처, 라벨) 쌍을 모델 학습 루프에 전달
        yield (feature, label)      # yield 사용 시 현재 처리중인 영상 한개만 메모리에 유지하여 성능 방어

In [4]:
from tensorflow.keras.utils import Sequence, to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import math
import numpy as np

class Dataloader(Sequence):
    def __init__(self, x_set, y_set, batch_size, shuffle=False):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.x) / self.batch_size)

    def __getitem__(self, idx):
        indices = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        # (주의) frame_extraction 함수는 기존에 정의된 것을 사용한다고 가정
        batch_x = [frame_extraction(self.x[i]) for i in indices]
        batch_y = [self.y[i] for i in indices]
        
        # GPU 충돌 방지를 위해 float32로 넘기면, 위에서 설정한 Policy가 알아서 FP16으로 변환해 연산합니다.
        return np.array(batch_x, dtype=np.float32), to_categorical(np.array(batch_y), num_classes=3)

    def on_epoch_end(self):
        self.indices = np.arange(len(self.x))
        if self.shuffle:
            np.random.shuffle(self.indices)

In [5]:
# ==========================================
# 1. 경로 설정
# ==========================================

BASE_PATH = '../이미지데이터'

def load_data_from_folder(base_dir):
    labels = []
    paths = []
    categories = ['신호위반', '중앙선침범', '진로변경위반']

    if not os.path.exists(base_dir):
        print(f"❌ 오류: 폴더를 찾을 수 없습니다 -> {base_dir}")
        return [], []

    print(f"📂 [{base_dir}] 데이터 로드 시작...")
    for label in categories:
        # 1. 카테고리 폴더가 바로 있는지 확인
        target_path = os.path.join(base_dir, label)
        
        # 2. 하위 폴더 탐색 (os.walk)
        count = 0
        for root, directories, files in os.walk(base_dir):
            # 경로 이름에 카테고리명(예: 신호위반)이 포함되어 있으면 수집
            if label in root: 
                has_image = False
                for file in files:
                    ext = file.split('.')[-1].lower()
                    if ext in ['jpg', 'jpeg', 'png']:
                        if root not in paths:
                            paths.append(root)
                            labels.append(categories.index(label))
                            has_image = True
                            count += 1
                # (중요) 한 폴더에 이미지가 여러 개여도 시퀀스는 1개로 쳐야 중복 집계 안 됨
                # 위 로직은 이미지 단위가 아니라 폴더 단위로 path를 넣으므로 정상.
        print(f"  -> '{label}' 데이터(폴더) 발견: {count}개")
    return paths, labels

# ==========================================
# 2. 데이터 로드 및 분할 (안전 장치 추가)
# ==========================================
train_source_dir = os.path.join(BASE_PATH, 'training')
val_source_dir = os.path.join(BASE_PATH, 'validation')

# 학습 데이터 로드
print("1. 학습 데이터 로드 중...")
train_paths, train_labels = load_data_from_folder(train_source_dir)

# 검증 데이터 로드
print("2. 검증 데이터 로드 중...")
val_all_paths, val_all_labels = load_data_from_folder(val_source_dir)

# -----------------------------------------------------------
# [중요] 데이터가 제대로 로드되었는지 확인 후 분할 진행
# -----------------------------------------------------------
if len(train_paths) == 0 or len(val_all_paths) == 0:
    print("\n🚨 [치명적 오류] 데이터를 찾지 못했습니다!")
    print(f"설정된 경로: {BASE_PATH}")
    print("폴더 경로가 정확한지, 해당 폴더 안에 'training'과 'validation' 폴더가 있는지 확인해주세요.")
else:
    # 검증 데이터를 반으로 쪼개서 테스트 데이터 생성
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        val_all_paths, val_all_labels, test_size=0.5, random_state=27, shuffle=True
    )

    print(f"\n✅ 데이터 준비 완료!")
    print(f"최종 데이터 개수: Train({len(train_paths)}), Val({len(val_paths)}), Test({len(test_paths)})")


    # ==========================================
    # 3. Dataloader 연결
    # ==========================================
    
    batch_size = 8
    train_dataset = Dataloader(train_paths, train_labels, batch_size, shuffle=True)
    val_dataset =  Dataloader(val_paths, val_labels, batch_size)
    test_dataset =  Dataloader(test_paths, test_labels, batch_size)

1. 학습 데이터 로드 중...
📂 [../이미지데이터/training] 데이터 로드 시작...
  -> '신호위반' 데이터(폴더) 발견: 2997개
  -> '중앙선침범' 데이터(폴더) 발견: 1998개
  -> '진로변경위반' 데이터(폴더) 발견: 1572개
2. 검증 데이터 로드 중...
📂 [../이미지데이터/validation] 데이터 로드 시작...
  -> '신호위반' 데이터(폴더) 발견: 527개
  -> '중앙선침범' 데이터(폴더) 발견: 400개
  -> '진로변경위반' 데이터(폴더) 발견: 271개

✅ 데이터 준비 완료!
최종 데이터 개수: Train(6567), Val(599), Test(599)


In [ ]:
from tensorflow.keras.layers import TimeDistributed, Dropout, LSTM, Dense, GlobalAveragePooling2D, Input, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Activation
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
import tensorflow as tf

def create_LRCN_model():
    # 1. 입력 설정
    inputs = Input(shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3))
    
    # 2. 특징 추출기 (CNN) - 필터 수를 줄여서 더 가볍게 만듦
    # Layer 1
    x = TimeDistributed(Conv2D(16, (3, 3), padding='same', kernel_regularizer=regularizers.l2(0.001)))(inputs)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Activation('relu'))(x)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(Dropout(0.25))(x)
    
    # Layer 2
    x = TimeDistributed(Conv2D(32, (3, 3), padding='same', kernel_regularizer=regularizers.l2(0.001)))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Activation('relu'))(x)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(Dropout(0.25))(x)
    
    # Layer 3
    x = TimeDistributed(Conv2D(64, (3, 3), padding='same', kernel_regularizer=regularizers.l2(0.001)))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Activation('relu'))(x)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(Dropout(0.25))(x)
    
    # ★★★ [여기가 핵심 수정] ★★★
    # Flatten()을 쓰면 파라미터가 폭발해서 과적합됩니다.
    # GlobalAveragePooling2D()로 교체하여 핵심 정보만 압축합니다.
    x = TimeDistributed(GlobalAveragePooling2D())(x)
    
    # 3. 시계열 분석 (LSTM)
    # 입력이 가벼워졌으므로 LSTM 유닛을 32나 64로 유지해도 충분합니다.
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(0.5)(x) # 마지막 강력한 규제
    
    # 4. 출력층
    outputs = Dense(len(categories), activation='softmax', dtype='float32')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# -------------------------------------------------------------
# 모델 생성 및 컴파일
# -------------------------------------------------------------
LRCN_model = create_LRCN_model()
LRCN_model.compile(
    loss='categorical_crossentropy', 
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), # 모델이 가벼워졌으므로 학습률을 기본값(0.001)으로 원복
    metrics=["accuracy"]
)
LRCN_model.summary()

# 4. 학습 시작
# 텐서보드 설정 (안 되어 있을 경우 대비)
if 'TensorB' not in locals():
    import datetime
    log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    TensorB = TensorBoard(log_dir=log_dir)

early_stopping_callback = EarlyStopping(
    monitor='val_loss', 
    patience=15, 
    mode='min', 
    restore_best_weights=True,
    verbose=1
)

checkpoint_callback = ModelCheckpoint(
    'best_model.h5', 
    monitor='val_loss', 
    save_best_only=True, 
    mode='min', 
    verbose=1
)

reduce_lr_callback = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=5, 
    min_lr=1e-6, 
    verbose=1
)

my_callbacks = [early_stopping_callback, TensorB, checkpoint_callback, reduce_lr_callback]

print("🚀 GPU 학습 시작")

LRCN_model_training_history2 = LRCN_model.fit(
    train_dataset,
    epochs=100,
    shuffle=True,
    validation_data=val_dataset,
    verbose=1,
    callbacks=my_callbacks,
    workers=4,
    use_multiprocessing=True
)



Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 50, 128, 128, 3   0         
                             )]                                  
                                                                 
 time_distributed (TimeDist  (None, 50, 128, 128, 16   448       
 ributed)                    )                                   
                                                                 
 time_distributed_1 (TimeDi  (None, 50, 128, 128, 16   64        
 stributed)                  )                                   
                                                                 
 time_distributed_2 (TimeDi  (None, 50, 128, 128, 16   0         
 stributed)                  )                                   
                                                                 
 time_distributed_3 (TimeDi  (None, 50, 64, 64, 16)    0     

I0000 00:00:1770731011.420929   75968 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


821/821 [==============================] - ETA: 0s - loss: 1.0437 - accuracy: 0.5033
Epoch 1: val_loss improved from inf to 1.01005, saving model to best_model.h5
821/821 [==============================] - 1182s 1s/step - loss: 1.0437 - accuracy: 0.5033 - val_loss: 1.0101 - val_accuracy: 0.5042 - lr: 0.0010


/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Epoch 2/100
821/821 [==============================] - ETA: 0s - loss: 1.0003 - accuracy: 0.5194
Epoch 2: val_loss did not improve from 1.01005
821/821 [==============================] - 1106s 1s/step - loss: 1.0003 - accuracy: 0.5194 - val_loss: 1.0366 - val_accuracy: 0.5025 - lr: 0.0010
Epoch 3/100
821/821 [==============================] - ETA: 0s - loss: 0.9766 - accuracy: 0.5389
Epoch 3: val_loss did not improve from 1.01005
821/821 [==============================] - 1102s 1s/step - loss: 0.9766 - accuracy: 0.5389 - val_loss: 1.0762 - val_accuracy: 0.4240 - lr: 0.0010
Epoch 4/100
821/821 [==============================] - ETA: 0s - loss: 0.9642 - accuracy: 0.5415
Epoch 4: val_loss improved from 1.01005 to 0.99521, saving model to best_model.h5
821/821 [==============================] - 1096s 1s/step - loss: 0.9642 - accuracy: 0.5415 - val_loss: 0.9952 - val_accuracy: 0.5109 - lr: 0.0010
Epoch 5/100
821/821 [==============================] - ETA: 0s - loss: 0.9519 - accuracy: 0.550

In [ ]:
def classifier(folder_path):
    categories = ['신호위반', '중앙선침범','진로변경위반']
    frame_list = frame_extraction(folder_path)
    predicted_labels_probabilities = LRCN_model.predict(np.expand_dims(frame_list, axis=0))[0]
    predicted_label = np.argmax(predicted_labels_probabilities)
    return print(f"탐지된 결과: {categories[predicted_label]}")

In [ ]:
def plot_metric(model_training_history, # 학습 중 기록된 loss, accuracy 데이터가 담긴 객체
                metric_name_1,
                metric_name_2,
                plot_name): # 그래프 제목
    '''
    This function will plot the metrics passed to it in a graph.
    Args:
        model_training_history: A history object containing a record of training and validation
                                loss values and metrics values at successive epochs
        metric_name_1:          The name of the first metric that needs to be plotted in the graph.
        metric_name_2:          The name of the second metric that needs to be plotted in the graph.
        plot_name:              The title of the graph.
    '''

    # hisory.history 딕셔너리에서 원하는 지표의 숫자 리스트 꺼내오기
    metric_value_1 = model_training_history.history[metric_name_1]
    metric_value_2 = model_training_history.history[metric_name_2]

    # 그래프 X축으로 학습된 횟수만큼 순서대로 숫자 생성
    epochs = range(len(metric_value_1))

    # Blue, Red 선 그리기
    plt.plot(epochs, metric_value_1, 'blue', label = metric_name_1)
    plt.plot(epochs, metric_value_2, 'red', label = metric_name_2)

    # Add title to the plot.
    plt.title(str(plot_name))

    # 학습용 및 검증용 설명 텍스트로 구분
    plt.legend()

In [ ]:
model_evaluation_history = LRCN_model.evaluate(val_dataset)

model_evaluation_loss, model_evaluation_accuracy = model_evaluation_history

date_time_format = '%Y_%m_%d__%H_%M_%S'
current_date_time_dt = dt.datetime.now()
current_date_time_string = dt.datetime.strftime(current_date_time_dt, date_time_format)

model_file_name = f'LRCN_model___Date_Time_{current_date_time_string}___Loss_{model_evaluation_loss}___Accuracy_{model_evaluation_accuracy}.h5'
save_dir = './lstm_model'
createDirectory(save_dir)
LRCN_model.save(os.path.join(save_dir, model_file_name))

In [ ]:
# =================================================================
# F1 Score 평가 (경로 '01.원천데이터' 추가 수정본)
# =================================================================

base_val_path = '../이미지데이터/validation'
target_eval_dir = os.path.join(base_val_path, '01.원천데이터') 

# 함수 정의 (기존 로직 유지)
def calculation_f1score(folder, true_positive=0, false_positive=0, false_negative=0):
    folder_name = os.path.basename(os.path.normpath(folder))
    
    try:
        positive_label = categories.index(folder_name)
    except ValueError:
        print(f"⚠️ 스킵: '{folder_name}' (카테고리 아님)")
        return folder_name, 0, 0, 0, 0, 0, 0

    real_label = positive_label
    folders = []
    
    # 이미지 파일이 있는 하위 폴더 탐색
    for root, directories, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                if root not in folders:
                    folders.append(root)
                    

    print(f"--- '{folder_name}' 평가 시작 (총 {len(folders)}개 시퀀스) ---")

    for test_folder in folders:
        frame_list = frame_extraction(test_folder)
        if len(frame_list) == 0: continue
            
        # 예측 (verbose=0으로 로그 숨김)
        predicted_labels_probabilities = LRCN_model.predict(np.expand_dims(frame_list, axis=0), verbose=0)[0]
        predicted_label = np.argmax(predicted_labels_probabilities)

        if predicted_label == positive_label and real_label == positive_label:
            true_positive += 1
        elif predicted_label == positive_label and real_label != positive_label:
            false_positive += 1
        elif predicted_label != positive_label and real_label == positive_label:
            false_negative += 1

    # 점수 계산
    if (true_positive + false_positive) == 0:
        precision = 0
    else:
        precision = true_positive / (true_positive + false_positive)
    
    if (true_positive + false_negative) == 0:
        recall = 0
    else:
        recall = true_positive / (true_positive + false_negative)
    
    try:
        f1_score = 2.0 * (precision * recall) / (precision + recall + 1e-7)
    except:
        f1_score = 0

    # 파일 저장
    with open('./lstm_f1_score.txt','a', encoding='utf-8') as f:
        f.write(f"{folder_name}, Total:{len(folders)}, TP:{true_positive}, FP:{false_positive}, FN:{false_negative}, F1:{round(f1_score,2)}\n")

    return folder_name, true_positive, false_positive, false_negative, precision, recall, f1_score

# ---------------------------------------------------------
# 실행 부분
# ---------------------------------------------------------
# 파일 초기화
with open('./lstm_f1_score.txt','w', encoding='utf-8') as f:
    f.write(f"Class, Count, TP, FP, FN, F1_score\n")

f1_scores_sum = 0
cnt = 0

print(f"📂 평가 데이터 경로: {target_eval_dir}")

for label in categories:
    # ../이미지데이터/validation/01.원천데이터/신호위반
    cls_path = os.path.join(target_eval_dir, label)
    
    if os.path.exists(cls_path):
        class_, tp, fp, fn, precision, recall, f1_score = calculation_f1score(cls_path)
        cnt += 1
        f1_scores_sum += f1_score
        print(f"👉 {class_} F1: {round(f1_score, 2)}\n")
    else:
        print(f"❌ 경로 없음: {cls_path}")

if cnt > 0:
    print(f"📊 최종 평균 F1 Score : {round(f1_scores_sum/cnt, 2)}")
else:
    print("❌ 평가할 폴더를 하나도 찾지 못했습니다. 경로를 다시 확인하세요.")

In [ ]:
# 학습이 끝난 모델로 테스트 진행
# LRCN_model = tf.keras.models.load_model('./lstm_model/LRCN_model___Date_Time_2023_03_01__04_35_27___Loss_0.01098685897886753___Accuracy_0.9961240291595459.h5')

In [ ]:
# # 이미 학습된 모델로 테스트 진행
# model_path = 'data/models/위반상황분류.h5'

# try:
#     # tf.keras 대신 tf_keras를 사용해 로드
#     LRCN_model = tf_keras.models.load_model(model_path, compile=False)
#     print("모델 로드 성공!")

#     # 모델 구조 확인 (제대로 불러와졌는지 체크)
#     LRCN_model.summary()

# except Exception as e:
#     print("모델 로드 실패:", e)

신호위반 테스트(적색신호시직진)

In [ ]:
path = "../이미지데이터/validation/01.원천데이터/신호위반/적색신호시좌회전/20230223_적색신호시좌회전_0000003598"

path_ = path
fig = plt.figure(figsize=(10,10)) # rows*cols 행렬의 i번째 subplot 생성
rows = 5
cols = 5
i = 1

xlabels = [f"{x}" if x!=0 else 'xlabel' for x in range(26) ]


for filename in os.listdir(path_)[:25]:
    filename = os.path.join(path_, filename)
    img = cv2.imread(filename)

    ax = fig.add_subplot(rows, cols, i)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_xlabel(xlabels[i])
    ax.set_xticks([]), ax.set_yticks([])
    i += 1

plt.show()
classifier(path)